# LeetCode 776 — Split BST bằng Python

Link đề: https://leetcode.com/problems/split-bst/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Search Tree, Recursion, Binary Tree

> **Bài Premium (🔒).** Đề trong notebook được viết theo hiểu biết về bài; nếu có tài khoản Premium, hãy **đối chiếu lại đề gốc** (nhất là các ràng buộc và cách LeetCode kiểm tra đáp án).

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def splitBST(self, root: Optional[TreeNode], target: int) -> List[Optional[TreeNode]]:
        ...
```

Ghi chú: bài **nhiều đáp án đúng về hình dạng cây** (đề cho phép nhiều cách tách miễn thoả điều kiện). Vì vậy notebook **không so cứng hình dạng** mà test bằng **hàm kiểm tra tính chất**: mỗi phần là BST hợp lệ, phần 1 gồm toàn giá trị `≤ target`, phần 2 toàn giá trị `> target`, hai phần chứa đúng các nút của cây gốc, và **quan hệ cha–con được giữ** theo đề. `TreeNode` cùng hàm dựng/xuất cây kiểu LeetCode được **tự dựng** trong notebook.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cắt một **cây tìm kiếm nhị phân (BST)** thành **hai BST**: một cây chứa các nút có giá trị `≤ target`, một cây chứa các nút `> target`, đồng thời **giữ càng nhiều cấu trúc cũ càng tốt** (nếu cha `p` và con `c` cùng thuộc một phần thì `c` vẫn là con của `p`).
2. Nắm các công cụ Python: `TreeNode`, đọc/ghi cây dạng danh sách theo tầng, duyệt **trung thứ tự (inorder)** của BST, **trả về hai giá trị** bằng list rồi gán `l, r = ...`, và **sao chép sâu** một cây để test hàm sửa cây tại chỗ.
3. Đi từ **dựng lại hai cây từ danh sách giá trị** → **đệ quy cắt tại chỗ (cắt rồi nối lại con trỏ)** → **bản lặp bằng hai "đuôi" giả (dummy)** → **bản không phá cây gốc (persistent)**.
4. Hiểu mấu chốt: so sánh `root.val` với `target` **quyết định hẳn** nút này và **cả một cây con** thuộc phần nào; chỉ cây con ở phía còn lại mới cần cắt tiếp — ta chỉ đi dọc **một đường** từ gốc xuống (độ dài `O(h)`).
5. Biết cách **test bài có nhiều đáp án đúng**: viết hàm kiểm tra tính chất (BST, ngưỡng, đủ nút, giữ quan hệ cha–con), và tránh các lỗi kinh điển: dùng `<` thay `<=`, quên nối lại con trỏ, nối sai phía, không cắt tiếp cây con còn lại.

## 2. Đề bài & ví dụ

Cho gốc `root` của một BST và một số nguyên `target`. Hãy **tách** cây thành **hai cây con**:

- cây thứ nhất gồm các nút có giá trị **nhỏ hơn hoặc bằng** `target`;
- cây thứ hai gồm các nút có giá trị **lớn hơn** `target`.

Không nhất thiết cây có nút với giá trị đúng bằng `target`.

Ngoài ra **phần lớn cấu trúc của cây gốc nên được giữ lại**. Chính xác: với mọi nút con `c` có cha `p` trong cây gốc, nếu sau khi tách cả hai **cùng thuộc một cây con** thì `c` vẫn phải có cha `p`.

Trả về **mảng gồm hai gốc** theo thứ tự `[gốc_cây_≤_target, gốc_cây_>_target]` (cây rỗng biểu diễn bằng `null`).

**Ví dụ 1:** `root = [4,2,6,1,3,5,7]`, `target = 2` ⇒ `[[2,1],[4,3,6,null,null,5,7]]`.

```
        4                  2          4
       / \                / \        / \
      2   6      ->      1   .      3   6
     / \ / \                            / \
    1  3 5  7                          5   7
```

**Ví dụ 2:** `root = [1]`, `target = 1` ⇒ `[[1],[]]`.

**Ràng buộc:**

- Số nút trong `[1, 50]`.
- `0 <= Node.val, target <= 1000`.

(Theo kiểu BST thông thường, các giá trị trong cây khác nhau; các test của notebook sinh cây với giá trị khác nhau.)

Cell dưới dựng `TreeNode`, `build_tree`, `tree_to_list` và vài hàm phụ.

In [1]:
from collections import deque
import random
import time
from typing import List, Optional

In [2]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

In [3]:
def show(root, indent=0, tag="root"):
    """In cây xoay ngang: mỗi nút một dòng, thụt lề theo độ sâu."""
    if root is None:
        print("    " * indent + f"{tag}: (rỗng)")
        return
    print("    " * indent + f"{tag}: {root.val}")
    if root.left or root.right:
        show(root.left, indent + 1, "L")
        show(root.right, indent + 1, "R")


ex1 = [4, 2, 6, 1, 3, 5, 7]
show(build_tree(ex1))
print(tree_to_list(build_tree(ex1)), tree_to_list(None))
expected1 = [[2, 1], [4, 3, 6, None, None, 5, 7]]
expected2 = [[1], []]

root: 4
    L: 2
        L: 1
        R: 3
    R: 6
        L: 5
        R: 7
[4, 2, 6, 1, 3, 5, 7] []


## 3. Kiến thức nền

### 3.1. Cây tìm kiếm nhị phân (BST) và duyệt trung thứ tự

**BST:** với mọi nút `x`, **mọi giá trị ở cây con trái `< x.val`** và **mọi giá trị ở cây con phải `> x.val`**. Hệ quả hay dùng: duyệt **trung thứ tự (inorder: trái → nút → phải)** cho dãy giá trị **tăng dần**. Ngược lại, nếu inorder không tăng chặt thì cây không phải BST (điều kiện cần; hàm `is_bst` dưới đây kiểm tra điều kiện đủ bằng cách truyền cận).

In [4]:
def inorder(root):
    return [] if root is None else inorder(root.left) + [root.val] + inorder(root.right)


def preorder(root):
    return [] if root is None else [root.val] + preorder(root.left) + preorder(root.right)


def is_bst(root, lo=float("-inf"), hi=float("inf")):
    """Mọi giá trị trong cây phải nằm CHẶT trong (lo, hi); trái nhận cận trên = val, phải nhận cận dưới = val."""
    if root is None:
        return True
    if not (lo < root.val < hi):
        return False
    return is_bst(root.left, lo, root.val) and is_bst(root.right, root.val, hi)


t = build_tree(ex1)
print(inorder(t), is_bst(t))                      # [1,2,3,4,5,6,7] True
bad = build_tree([4, 2, 6, 1, 5])                 # 5 nằm bên trái của 4 nhưng 5 > 4  -> không phải BST
print(inorder(bad), is_bst(bad))

[1, 2, 3, 4, 5, 6, 7] True
[1, 2, 5, 4, 6] False


### 3.2. So sánh với `target` quyết định cả một cây con

Giả sử xét nút `x`:

- Nếu `x.val ≤ target`: `x` thuộc **phần 1**; mọi nút ở cây con **trái** của `x` (nhỏ hơn `x`) cũng `≤ target` ⇒ **cả cây con trái thuộc phần 1** (không cần xét nữa). Chỉ cây con **phải** (có thể chứa cả giá trị `≤` lẫn `>` target) cần cắt tiếp.
- Nếu `x.val > target`: `x` thuộc **phần 2**; mọi nút ở cây con **phải** (lớn hơn `x`) cũng `> target` ⇒ **cả cây con phải thuộc phần 2**. Chỉ cây con **trái** cần cắt tiếp.

Vì thế mỗi bước ta chỉ đi xuống **một** phía ⇒ dọc theo **một đường** từ gốc, độ dài tối đa `h`.

### 3.3. Hàm trả về hai giá trị

Python cho hàm trả list/tuple hai phần tử và **gán song song**: `l, r = dfs(node)`. Ta sẽ dùng hàm `dfs(node)` trả `[phần_≤, phần_>]` của cây gốc `node`.

In [5]:
def min_max(a):
    return [min(a), max(a)]


lo, hi = min_max([3, 1, 4, 1, 5])
print(lo, hi)

1 5


### 3.4. Cắt và nối lại con trỏ

Nối/cắt cây = gán lại thuộc tính `left`/`right`. Ví dụ, "cắt" cây con phải của `x` rồi gắn một cây khác vào: `x.right = other`. Chú ý: **gán lại làm mất tham chiếu cũ** (nếu không giữ ở đâu thì cây con cũ bị mất). Vì vậy ta phải lưu cây con cần dùng vào biến trước khi gán.

In [6]:
x = TreeNode(5, TreeNode(3), TreeNode(8))
old_right = x.right                   # lưu lại trước khi cắt
x.right = None                        # cắt
print(tree_to_list(x), "| cây con vừa cắt:", tree_to_list(old_right))
x.right = old_right                   # nối lại
print(tree_to_list(x))

[5, 3] | cây con vừa cắt: [8]
[5, 3, 8]


### 3.5. Sao chép sâu cây để test hàm "sửa tại chỗ"

Lời giải đệ quy **thay đổi trực tiếp** các con trỏ của cây gốc. Để so sánh kết quả với cây gốc ban đầu (kiểm tra "giữ quan hệ cha–con"), ta cần **một bản sao** của cây gốc trước khi gọi hàm. `copy_tree` dựng một cây mới có cùng hình dạng và giá trị.

In [7]:
def copy_tree(root):
    return None if root is None else TreeNode(root.val, copy_tree(root.left), copy_tree(root.right))


a = build_tree(ex1)
b = copy_tree(a)
a.left = None                         # sửa a không ảnh hưởng b
print(tree_to_list(a), tree_to_list(b))

[4, None, 6, 5, 7] [4, 2, 6, 1, 3, 5, 7]


### 3.6. Bảng "cha của mỗi giá trị" (để kiểm tra quan hệ cha–con)

Vì các giá trị trong cây khác nhau, ta có thể nhận diện nút bằng **giá trị**. `parent_map(root)` trả từ điển `giá_trị -> (giá_trị_cha, phía)` với `phía` là `"L"` hoặc `"R"`; gốc có cha `None`. Hàm này dùng cho bộ kiểm tra ở mục 12.

In [8]:
def parent_map(root):
    pm = {}

    def go(node, parent, side):
        if node is None:
            return
        pm[node.val] = (parent, side)
        go(node.left, node.val, "L")
        go(node.right, node.val, "R")

    go(root, None, None)
    return pm


print(parent_map(build_tree(ex1)))

{4: (None, None), 2: (4, 'L'), 1: (2, 'L'), 3: (2, 'R'), 6: (4, 'R'), 5: (6, 'L'), 7: (6, 'R')}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm tay ví dụ 1: `4,2,6,1,3,5,7` với `target = 2`.

- Phần 1 phải chứa mọi nút `≤ 2`: `{1, 2}`. Phần 2 chứa `{3, 4, 5, 6, 7}`.
- Nhìn cây gốc: `4` > 2 nên `4` ở phần 2, **kéo theo cả cây con phải** `6,5,7` (đều lớn hơn 4 > 2) ở phần 2. Chỉ cây con **trái** của `4` (tức cây gốc `2`) cần xem tiếp.
- `2 ≤ 2` ⇒ `2` ở phần 1 cùng **cả cây con trái** `1`. Chỉ cây con **phải** của `2` (nút `3`) cần xem tiếp.
- `3 > 2` ⇒ `3` thuộc phần 2; cây con trái của `3` rỗng.
- Kết quả: phần 1 = `2` với con trái `1`. Phần 2 = `4` (đã mất con trái `2`, nay con trái là `3`) với con phải `6,5,7`. Quan hệ cha–con được giữ: `2→1` (cùng phần 1), `4→6`, `6→5`, `6→7` (cùng phần 2). Quan hệ `4→2` (khác phần), `2→3` (khác phần) bị cắt.

**Hình dung:** đi dọc đường từ gốc xuống theo so sánh với `target`; mỗi nút trên đường "chọn phe". Các cây con "không nằm trên đường" đi theo phe của nút cha (không cần động tới).

Ví dụ nhỏ khác: `[1]`, `target = 1` ⇒ `1 ≤ 1` thuộc phần 1; phần 2 rỗng ⇒ `[[1], []]`. Nếu `target = 0` ⇒ `1 > 0` thuộc phần 2; phần 1 rỗng ⇒ `[[], [1]]`.

## 5. Cách 1 — Brute Force / baseline: lấy danh sách giá trị rồi **dựng lại** hai cây

**Ý tưởng:** không động tới cây gốc. Duyệt **tiền thứ tự (preorder)** cây gốc để có dãy giá trị; chia dãy thành hai nhóm (`≤ target` và `> target`) **giữ nguyên thứ tự**; rồi **chèn từng giá trị vào một BST mới** theo thứ tự đó (hàm `insert` chuẩn của BST).

Vì chèn theo tiền thứ tự (cha trước con) nên hình dạng của mỗi cây được dựng lại **bám theo cây gốc** (xem lập luận ở mục 10). Cách này dễ nghĩ nhưng tạo **nút mới** và tốn `O(n · h)`.

In [9]:
def insert_bst(root, val):
    """Chèn val vào BST (lặp). Trả gốc."""
    node = TreeNode(val)
    if root is None:
        return node
    cur = root
    while True:
        if val < cur.val:
            if cur.left is None:
                cur.left = node
                return root
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = node
                return root
            cur = cur.right


def split_rebuild(root, target):
    """Cách 1: preorder -> chia theo target -> dựng lại hai BST bằng chèn tuần tự (nút mới)."""
    small = big = None
    for v in preorder(root):
        if v <= target:
            small = insert_bst(small, v)
        else:
            big = insert_bst(big, v)
    return [small, big]


res = split_rebuild(build_tree(ex1), 2)
print([tree_to_list(r) for r in res])
assert [tree_to_list(r) for r in res] == expected1
res = split_rebuild(build_tree([1]), 1)
print([tree_to_list(r) for r in res])
assert [tree_to_list(r) for r in res] == expected2

[[2, 1], [4, 3, 6, None, None, 5, 7]]
[[1], []]


**Độ phức tạp:** duyệt `O(n)`; mỗi lần chèn tốn `O(h')` (`h'` = chiều cao cây đang dựng ≤ `n`) ⇒ **`O(n · h)`** (tệ nhất `O(n²)`); bộ nhớ `O(n)` cho các nút mới (cộng ngăn xếp đệ quy của `preorder`).

**Nút thắt:** (1) tạo **toàn bộ nút mới** dù chỉ cần "cắt một đường"; (2) mỗi chèn đi lại từ gốc. Ta chỉ cần động tới các nút **trên một đường duy nhất** từ gốc, nên có thể cắt và nối lại con trỏ **tại chỗ** (mục 6).

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `dfs(node)` trả `[phần_≤, phần_>]` của cây gốc `node`. Hai trường hợp (dựa vào mục 3.2):

**Trường hợp `node.val ≤ target`:** `node` thuộc phần 1, cùng **toàn bộ cây con trái**.

- Cắt cây con **phải** bằng `l, r = dfs(node.right)` (`l` = phần `≤` của cây con phải, `r` = phần `>`).
- Nối `l` làm con phải mới của `node`: `node.right = l` — vì `l` toàn giá trị `≤ target` nhưng lớn hơn `node.val` (do nằm bên phải), nên `l` thuộc phần 1 và hợp lệ làm con phải.
- Trả `[node, r]`.

**Trường hợp `node.val > target`:** đối xứng.

- `l, r = dfs(node.left)`.
- `node.left = r` (`r` toàn giá trị `> target` nhưng nhỏ hơn `node.val`).
- Trả `[l, node]`.

**Cơ sở:** `dfs(None) = [None, None]`.

Mỗi bước đi xuống **một** con ⇒ tổng `O(h)` thời gian, và chỉ **ghi lại** một con trỏ mỗi nút trên đường đi. Phần còn lại của cây giữ nguyên ⇒ **cấu trúc được giữ tối đa**.

Ta sẽ xem: Cách 2 (đệ quy tại chỗ — chính thức), Cách 3 (bản lặp, hai "đuôi" giả), Cách 4 (bản **không phá** cây gốc: tạo nút mới chỉ dọc đường cắt, chia sẻ phần còn lại).

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

- Cách 1: dựng lại hai cây từ preorder (mục 5).
- Cách 2: đệ quy cắt và nối con trỏ tại chỗ (**chính thức**).
- Cách 3: bản lặp: đi dọc đường từ gốc xuống, nối vào hai "đuôi" giả.
- Cách 4: bản persistent — không sửa cây gốc, chỉ tạo nút mới dọc theo đường cắt.

---

### Cách 2 — Đệ quy cắt tại chỗ

Giải thích từng đoạn (khớp mục 6):

1. `dfs(None)` trả `[None, None]`.
2. Nếu `root.val <= target`: `l, r = dfs(root.right)`; `root.right = l`; trả `[root, r]`.
3. Ngược lại: `l, r = dfs(root.left)`; `root.left = r`; trả `[l, root]`.
4. Hàm chính trả `dfs(root)`.

In [10]:
def split_recursive(root, target):
    """Cách 2: đệ quy, sửa con trỏ tại chỗ."""
    def dfs(node):
        if node is None:
            return [None, None]
        if node.val <= target:
            l, r = dfs(node.right)
            node.right = l
            return [node, r]
        else:
            l, r = dfs(node.left)
            node.left = r
            return [l, node]

    return dfs(root)


res = split_recursive(build_tree(ex1), 2)
print([tree_to_list(r) for r in res])
assert [tree_to_list(r) for r in res] == expected1
res = split_recursive(build_tree([1]), 1)
assert [tree_to_list(r) for r in res] == expected2

[[2, 1], [4, 3, 6, None, None, 5, 7]]


**Code nộp LeetCode cho Cách 2:**

```python
class Solution:
    def splitBST(self, root: Optional[TreeNode], target: int) -> List[Optional[TreeNode]]:
        def dfs(root):
            if root is None:
                return [None, None]
            if root.val <= target:
                l, r = dfs(root.right)
                root.right = l
                return [root, r]
            else:
                l, r = dfs(root.left)
                root.left = r
                return [l, root]

        return dfs(root)
```

**Độ phức tạp:** thời gian `O(h)` (chỉ đi dọc một đường; tệ nhất `O(n)` khi cây lệch); bộ nhớ `O(h)` cho ngăn xếp đệ quy; **không tạo nút mới**.

---

### Cách 3 — Bản lặp với hai "đuôi" giả (dummy)

Giải thích từng đoạn:

1. Hai nút giả: `small_dummy` (đầu phần 1) và `big_dummy` (đầu phần 2). Hai con trỏ "đuôi": `small_tail = small_dummy`, `big_tail = big_dummy`.
2. Cho `cur = root`. Lặp khi `cur` khác rỗng:
   - Nếu `cur.val <= target`: nút thuộc phần 1. Nối `small_tail.right = cur`, `small_tail = cur`, rồi **tiếp tục xuống bên phải**: `nxt = cur.right`, và cắt `cur.right = None` (sẽ được nối lại ở lần sau nếu còn nút phần 1 trong cây con phải).
   - Ngược lại: nối `big_tail.left = cur`, `big_tail = cur`; tiếp tục xuống bên **trái**: `nxt = cur.left`; cắt `cur.left = None`.
   - `cur = nxt`.
3. Kết quả: `[small_dummy.right, big_dummy.left]`.

Vì nút phần 1 tiếp theo (nếu có) luôn nằm ở cây con **phải** và **lớn hơn** nút hiện tại, nối nó làm con **phải** của đuôi là đúng; nút phần 2 tiếp theo nằm ở cây con **trái** và **nhỏ hơn**, nối làm con **trái** của đuôi là đúng.

In [11]:
def split_iterative(root, target):
    """Cách 3: lặp, hai đuôi giả; phần 1 nối qua .right, phần 2 nối qua .left."""
    small_dummy, big_dummy = TreeNode(), TreeNode()
    small_tail, big_tail = small_dummy, big_dummy
    cur = root
    while cur:
        if cur.val <= target:
            nxt = cur.right                  # lưu phần còn phải cắt
            cur.right = None                 # cắt: sẽ nối lại khi tìm được nút phần 1 kế tiếp
            small_tail.right = cur
            small_tail = cur
        else:
            nxt = cur.left
            cur.left = None
            big_tail.left = cur
            big_tail = cur
        cur = nxt
    return [small_dummy.right, big_dummy.left]


res = split_iterative(build_tree(ex1), 2)
print([tree_to_list(r) for r in res])
assert [tree_to_list(r) for r in res] == expected1
res = split_iterative(build_tree([1]), 1)
assert [tree_to_list(r) for r in res] == expected2

[[2, 1], [4, 3, 6, None, None, 5, 7]]


**Code nộp LeetCode cho Cách 3:**

```python
class Solution:
    def splitBST(self, root: Optional[TreeNode], target: int) -> List[Optional[TreeNode]]:
        small_dummy, big_dummy = TreeNode(), TreeNode()
        small_tail, big_tail = small_dummy, big_dummy
        cur = root
        while cur:
            if cur.val <= target:
                nxt = cur.right
                cur.right = None
                small_tail.right = cur
                small_tail = cur
            else:
                nxt = cur.left
                cur.left = None
                big_tail.left = cur
                big_tail = cur
            cur = nxt
        return [small_dummy.right, big_dummy.left]
```

**Độ phức tạp:** `O(h)` thời gian, **`O(1)`** bộ nhớ phụ (không đệ quy), không tạo nút mới ngoài hai nút giả.

---

### Cách 4 — Bản persistent: không sửa cây gốc

Có lúc ta **không được phá** cây gốc (ví dụ cần giữ lịch sử/phiên bản). Cách làm: giống Cách 2 nhưng thay vì sửa `node.right = l`, ta **tạo nút mới** `TreeNode(node.val, node.left, l)` chỉ cho các nút **trên đường cắt**; mọi cây con còn lại được **chia sẻ** (dùng chung) với cây gốc. Cây gốc nguyên vẹn.

Giải thích từng đoạn:

1. `dfs(None) = [None, None]`.
2. Nếu `node.val <= target`: `l, r = dfs(node.right)`; phần 1 = nút mới `TreeNode(node.val, node.left, l)`; phần 2 = `r`.
3. Ngược lại: `l, r = dfs(node.left)`; phần 1 = `l`; phần 2 = nút mới `TreeNode(node.val, r, node.right)`.

In [12]:
def split_persistent(root, target):
    """Cách 4: không sửa cây gốc; chỉ tạo nút mới dọc đường cắt, chia sẻ các cây con còn lại."""
    if root is None:
        return [None, None]
    if root.val <= target:
        l, r = split_persistent(root.right, target)
        return [TreeNode(root.val, root.left, l), r]
    l, r = split_persistent(root.left, target)
    return [l, TreeNode(root.val, r, root.right)]


orig = build_tree(ex1)
res = split_persistent(orig, 2)
print([tree_to_list(r) for r in res])
print("cây gốc vẫn nguyên:", tree_to_list(orig))
assert [tree_to_list(r) for r in res] == expected1 and tree_to_list(orig) == ex1
res = split_persistent(build_tree([1]), 1)
assert [tree_to_list(r) for r in res] == expected2

[[2, 1], [4, 3, 6, None, None, 5, 7]]
cây gốc vẫn nguyên: [4, 2, 6, 1, 3, 5, 7]


**Code nộp LeetCode cho Cách 4:**

```python
class Solution:
    def splitBST(self, root: Optional[TreeNode], target: int) -> List[Optional[TreeNode]]:
        if root is None:
            return [None, None]
        if root.val <= target:
            l, r = self.splitBST(root.right, target)
            return [TreeNode(root.val, root.left, l), r]
        l, r = self.splitBST(root.left, target)
        return [l, TreeNode(root.val, r, root.right)]
```

**Độ phức tạp:** `O(h)` thời gian; bộ nhớ `O(h)` (ngăn xếp đệ quy + `O(h)` nút mới, vì chỉ tạo nút trên đường cắt). Cây gốc **không bị thay đổi**.

## 8. TRACE từng bước

### 8.1. Trace Cách 2 với ví dụ 1 (`target = 2`)

Cây: `4(2(1,3), 6(5,7))`.

| gọi | node | so với 2 | hành động | trả về |
|---|---|---|---|---|
| `dfs(4)` | 4 | `4 > 2` | cắt cây trái: `l, r = dfs(2)` | (chờ) |
| `dfs(2)` | 2 | `2 ≤ 2` | cắt cây phải: `l, r = dfs(3)` | (chờ) |
| `dfs(3)` | 3 | `3 > 2` | cắt cây trái: `l, r = dfs(None) = [None, None]`; `3.left = r = None` | `[None, 3]` |
| quay lại `dfs(2)` | 2 | — | `l = None`, `r = 3`; `2.right = l = None` | `[2, 3]` |
| quay lại `dfs(4)` | 4 | — | `l = 2`, `r = 3`; `4.left = r = 3` | `[2, 4]` |

Kết quả: phần 1 = `2(1)`, phần 2 = `4(3, 6(5,7))` ⇒ `[[2,1],[4,3,6,null,null,5,7]]` ✓.

In [13]:
def trace_split(root, target):
    def dfs(node, depth):
        pad = "  " * depth
        if node is None:
            print(f"{pad}dfs(None) -> [None, None]")
            return [None, None]
        side = "<=" if node.val <= target else ">"
        print(f"{pad}dfs({node.val}): {node.val} {side} {target}")
        if node.val <= target:
            l, r = dfs(node.right, depth + 1)
            node.right = l
            print(f"{pad}  về {node.val}: right = {l.val if l else None}; trả [{node.val}, {r.val if r else None}]")
            return [node, r]
        l, r = dfs(node.left, depth + 1)
        node.left = r
        print(f"{pad}  về {node.val}: left = {r.val if r else None}; trả [{l.val if l else None}, {node.val}]")
        return [l, node]

    return dfs(root, 0)


res = trace_split(build_tree(ex1), 2)
print([tree_to_list(r) for r in res])
assert [tree_to_list(r) for r in res] == expected1

dfs(4): 4 > 2
  dfs(2): 2 <= 2
    dfs(3): 3 > 2
      dfs(None) -> [None, None]
      về 3: left = None; trả [None, 3]
    về 2: right = None; trả [2, 3]
  về 4: left = 3; trả [2, 4]
[[2, 1], [4, 3, 6, None, None, 5, 7]]


### 8.2. Trace với `target = 5` trên cùng cây (đường cắt dài hơn)

`dfs(4)`: `4 ≤ 5` ⇒ cắt cây phải `dfs(6)`: `6 > 5` ⇒ cắt cây trái `dfs(5)`: `5 ≤ 5` ⇒ cắt cây phải `dfs(None) = [None, None]`, `5.right = None`, trả `[5, None]`. Về `6`: `6.left = r = None`, trả `[5, 6]`. Về `4`: `4.right = l = 5`, trả `[4, 6]`.

Phần 1 = `4(2(1,3), 5)`; phần 2 = `6(., 7)`. Đối chiếu quan hệ cha–con: `4→2`, `2→1`, `2→3` giữ (cùng phần 1); `4→6` bị cắt (khác phần); `6→5` bị cắt; `6→7` giữ (cùng phần 2); `4→5` là quan hệ **mới** (5 trước kia là cháu của 4 qua 6; điều này được phép vì `6` không còn thuộc phần của `5`).

In [14]:
res = trace_split(build_tree(ex1), 5)
print([tree_to_list(r) for r in res])
assert [tree_to_list(r) for r in res] == [[4, 2, 5, 1, 3], [6, None, 7]]

dfs(4): 4 <= 5
  dfs(6): 6 > 5
    dfs(5): 5 <= 5
      dfs(None) -> [None, None]
      về 5: right = None; trả [5, None]
    về 6: left = None; trả [5, 6]
  về 4: right = 5; trả [4, 6]
[[4, 2, 5, 1, 3], [6, None, 7]]


### 8.3. Trace Cách 3 (bản lặp) với `target = 2`

`cur = 4`: `4 > 2` ⇒ `nxt = 4.left = 2`; `4.left = None`; `big_tail.left = 4`; `big_tail = 4`; `cur = 2`.
`cur = 2`: `2 ≤ 2` ⇒ `nxt = 2.right = 3`; `2.right = None`; `small_tail.right = 2`; `small_tail = 2`; `cur = 3`.
`cur = 3`: `3 > 2` ⇒ `nxt = 3.left = None`; `3.left = None`; `big_tail.left = 3` (tức `4.left = 3`); `big_tail = 3`; `cur = None`.
Kết quả: `[small_dummy.right = 2, big_dummy.left = 4]` với `2(1)` và `4(3, 6(5,7))` ✓.

In [15]:
def trace_iterative(root, target):
    small_dummy, big_dummy = TreeNode(), TreeNode()
    small_tail, big_tail = small_dummy, big_dummy
    cur = root
    while cur:
        if cur.val <= target:
            print(f"cur={cur.val}: {cur.val} <= {target} -> nối vào đuôi phần 1 (đuôi hiện tại: {small_tail.val if small_tail is not small_dummy else 'dummy'}), đi xuống PHẢI")
            nxt = cur.right
            cur.right = None
            small_tail.right = cur
            small_tail = cur
        else:
            print(f"cur={cur.val}: {cur.val} > {target} -> nối vào đuôi phần 2 (đuôi hiện tại: {big_tail.val if big_tail is not big_dummy else 'dummy'}), đi xuống TRÁI")
            nxt = cur.left
            cur.left = None
            big_tail.left = cur
            big_tail = cur
        cur = nxt
    return [small_dummy.right, big_dummy.left]


res = trace_iterative(build_tree(ex1), 2)
print([tree_to_list(r) for r in res])
assert [tree_to_list(r) for r in res] == expected1

cur=4: 4 > 2 -> nối vào đuôi phần 2 (đuôi hiện tại: dummy), đi xuống TRÁI
cur=2: 2 <= 2 -> nối vào đuôi phần 1 (đuôi hiện tại: dummy), đi xuống PHẢI
cur=3: 3 > 2 -> nối vào đuôi phần 2 (đuôi hiện tại: 4), đi xuống TRÁI
[[2, 1], [4, 3, 6, None, None, 5, 7]]


## 9. Code nộp LeetCode

Cách 2 (chính thức): đúng `class Solution`, phương thức `splitBST(self, root: Optional[TreeNode], target: int) -> List[Optional[TreeNode]]`.

In [16]:
class Solution:
    def splitBST(self, root: Optional[TreeNode], target: int) -> List[Optional[TreeNode]]:
        def dfs(root):
            if root is None:
                return [None, None]
            if root.val <= target:
                l, r = dfs(root.right)
                root.right = l
                return [root, r]
            else:
                l, r = dfs(root.left)
                root.left = r
                return [l, root]

        return dfs(root)


sol = Solution()
res = sol.splitBST(build_tree([4, 2, 6, 1, 3, 5, 7]), 2)
print([tree_to_list(r) for r in res])           # [[2, 1], [4, 3, 6, None, None, 5, 7]]
res = sol.splitBST(build_tree([1]), 1)
print([tree_to_list(r) for r in res])           # [[1], []]

[[2, 1], [4, 3, 6, None, None, 5, 7]]
[[1], []]


## 10. Vì sao đúng

Gọi `dfs(x)` trả `[A, B]` với kỳ vọng: `A` là BST gồm đúng các nút `≤ target` của cây gốc `x`, `B` là BST gồm đúng các nút `> target`, và quan hệ cha–con bên trong `x` được giữ nếu cả hai cùng phần. Quy nạp theo chiều cao.

- **Cơ sở:** `x = None` ⇒ `[None, None]` ✓.
- **`x.val ≤ target`:** cây con trái của `x` toàn giá trị `< x.val ≤ target` ⇒ nằm hết ở phần 1 và **không đổi** (quan hệ cha–con bên trong nó và với `x` đều giữ: `x → x.left` giữ). Theo giả thiết quy nạp, `dfs(x.right) = [l, r]` với `l` = nút `≤ target` của cây phải, `r` = nút `> target`. Ta đặt `x.right = l` và trả `[x, r]`:
  - **BST:** `l` ⊂ cây con phải gốc nên mọi giá trị trong `l` `> x.val` ✓; cây con trái `< x.val` ✓.
  - **Ngưỡng:** `x` và toàn bộ `l`, cây trái đều `≤ target`; `r` đều `> target` ✓.
  - **Đủ nút:** mỗi nút của cây `x` ở đúng một nơi ✓.
  - **Quan hệ cha–con:** `x → x.left` giữ; `x → x.right` chỉ giữ nếu gốc của `x.right` thuộc `l` — nếu gốc của `l` chính là con phải cũ thì giữ, nếu con phải cũ thuộc `r` thì khác phần nên được phép đổi.
- **`x.val > target`:** đối xứng: cây phải cố định ở phần 2, `x.left = r`, trả `[l, x]`.

**Vì sao ở Cách 3 (lặp) vẫn đúng?** Nó là phép khử đệ quy của Cách 2: với phần 1, mỗi nút `≤ target` nối vào **con phải** của nút phần 1 trước đó (nút phần 1 kế tiếp luôn nằm ở cây con phải và lớn hơn); với phần 2, nối vào **con trái** (nhỏ hơn).

**Cách 1 giữ quan hệ cha–con:** chèn theo preorder; với cặp (cha `p`, con `c`) cùng phần, `p` đứng trước `c`, và `c` là **nút đầu tiên** (theo preorder) của cây con tương ứng của `p` thuộc phần đó (mọi nút khác của cây con ấy nằm sau `c` trong preorder), nên khi chèn `c`, nó trở thành con trực tiếp của `p` đúng phía.

**Cách 4:** cùng logic Cách 2 nhưng dùng nút mới cho nút trên đường cắt; cây con không nằm trên đường được chia sẻ nguyên vẹn.

Ta kiểm chứng toàn bộ bằng **hàm kiểm tra tính chất** trên hàng nghìn cây ngẫu nhiên ở mục 12.

## 11. Phân tích Time / Space Complexity

`n` = số nút, `h` = chiều cao cây.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Dựng lại từ preorder | `O(n · h)` | `O(n)` | tạo nút mới, oracle thứ hai |
| 2. Đệ quy tại chỗ | `O(h)` | `O(h)` | **nộp bài**, không tạo nút mới |
| 3. Lặp, hai đuôi giả | `O(h)` | `O(1)` | không đệ quy |
| 4. Persistent | `O(h)` | `O(h)` | không phá cây gốc |

Với `n ≤ 50` mọi cách đều tức thì; điểm của bài là **thao tác con trỏ đúng** và **chỉ đi một đường** (không duyệt hết `n` nút).

## 12. Bộ test

Vì bài **nhiều đáp án đúng về hình dạng**, ta không so cứng cây kết quả (trừ các ví dụ đề, nơi mọi cách của ta cho đúng output mẫu). Bộ kiểm tra `check_split(original, target, result)` xác nhận:

1. `result` có đúng hai phần tử.
2. Cả hai phần là **BST hợp lệ** (cận chặt).
3. Phần 1 chỉ gồm giá trị **`≤ target`**; phần 2 chỉ gồm giá trị **`> target`** (đúng ngưỡng).
4. Hợp hai phần chứa **đúng** các giá trị của cây gốc (không mất, không thừa, không trùng).
5. **Quan hệ cha–con được giữ:** với mọi cặp (cha `p`, con `c`) của cây gốc, nếu `p` và `c` cùng một phần thì `c` vẫn là con của `p` (cùng phía).

Hàm sửa cây tại chỗ phải được gọi trên **bản sao** (`copy_tree`) để vẫn còn cây gốc nguyên vẹn đem so. Ngoài ra: ví dụ đề (so output mẫu), edge case (target nhỏ hơn mọi giá trị, lớn hơn mọi giá trị, bằng giá trị gốc, cây một nút, cây lệch), ngẫu nhiên có seed đúng ràng buộc (≤ 50 nút, giá trị 0..1000).

In [17]:
def check_split(original, target, result):
    """Kiểm tra tính chất của một phép tách. `original` là bản sao CHƯA bị sửa của cây gốc."""
    assert isinstance(result, (list, tuple)) and len(result) == 2, "phải trả đúng hai gốc"
    A, B = result
    ia, ib = inorder(A), inorder(B)
    assert is_bst(A), "phần 1 không phải BST"
    assert is_bst(B), "phần 2 không phải BST"
    assert all(v <= target for v in ia), "phần 1 có giá trị > target"
    assert all(v > target for v in ib), "phần 2 có giá trị <= target"
    assert sorted(ia + ib) == sorted(inorder(original)), "tập nút không khớp cây gốc (mất/thừa/trùng)"
    po, pa, pb = parent_map(original), parent_map(A), parent_map(B)
    part = {v: "A" for v in ia}
    part.update({v: "B" for v in ib})
    for v, (p, side) in po.items():
        if p is not None and part[v] == part[p]:
            new = (pa if part[v] == "A" else pb)[v]
            assert new == (p, side), f"mất quan hệ cha-con: {p} -> {v} ({side}) nay là {new}"
    return True


# Chạy thử bộ kiểm tra với kết quả đúng và vài kết quả sai
orig = build_tree(ex1)
assert check_split(orig, 2, split_recursive(copy_tree(orig), 2))
for name, fake in [("sai ngưỡng", [build_tree([2, 1, 3]), build_tree([4, None, 6, None, None, 5, 7])]),
                   ("mất nút", [build_tree([2, 1]), build_tree([4, None, 6])])]:
    try:
        check_split(orig, 2, fake)
        print(name, ": KHÔNG bắt được (lỗi!)")
    except AssertionError as e:
        print(f"{name}: bộ kiểm tra bắt được ->", e)

sai ngưỡng: bộ kiểm tra bắt được -> phần 1 có giá trị > target
mất nút: bộ kiểm tra bắt được -> tập nút không khớp cây gốc (mất/thừa/trùng)


In [18]:
IMPLS = [split_rebuild, split_recursive, split_iterative, split_persistent, lambda r, t: Solution().splitBST(r, t)]
NAMES = ["rebuild", "recursive", "iterative", "persistent", "Solution"]


def run_checked(f, values, target):
    """Dựng cây, gọi f trên BẢN SAO, kiểm tra tính chất so với cây gốc nguyên vẹn. Trả hai danh sách kết quả."""
    original = build_tree(values)
    result = f(copy_tree(original), target)
    check_split(original, target, result)
    return [tree_to_list(r) for r in result]


# 1) Ví dụ đề: output mẫu cũng khớp CỨNG vì mọi cách của ta cho đúng hình dạng đó
for f, name in zip(IMPLS, NAMES):
    assert run_checked(f, ex1, 2) == expected1, name
    assert run_checked(f, [1], 1) == expected2, name
print("ví dụ đề: cả 5 cách khớp output mẫu và qua bộ kiểm tra")

# 2) Edge case, kiểm tra bằng tính chất
edge = [
    ([5], 3, [[], [5]]),                   # chỉ một nút, target nhỏ hơn
    ([5], 5, [[5], []]),                   # target bằng giá trị
    ([5], 9, [[5], []]),                   # target lớn hơn
    ([4, 2, 6, 1, 3, 5, 7], 0, [[], ex1]),         # target nhỏ hơn mọi giá trị -> không đổi
    ([4, 2, 6, 1, 3, 5, 7], 7, [ex1, []]),        # target >= mọi giá trị
    ([4, 2, 6, 1, 3, 5, 7], 4, [[4, 2, None, 1, 3], [6, 5, 7]]),   # target bằng giá trị gốc
    ([4, 2, 6, 1, 3, 5, 7], 5, [[4, 2, 5, 1, 3], [6, None, 7]]),
    ([4, 2, 6, 1, 3, 5, 7], 3, None),      # chỉ kiểm tra tính chất
    ([1, None, 2, None, 3, None, 4], 2, None),     # cây lệch phải
    ([4, 3, None, 2, None, 1], 2, None),            # cây lệch trái
]
for values, target, expected in edge:
    for f, name in zip(IMPLS, NAMES):
        got = run_checked(f, values, target)
        if expected is not None:
            assert got == expected, (name, values, target, got, expected)
print(len(edge), "ca biên: qua bộ kiểm tra (và khớp kết quả nếu đã tính tay chắc chắn)")

ví dụ đề: cả 5 cách khớp output mẫu và qua bộ kiểm tra
10 ca biên: qua bộ kiểm tra (và khớp kết quả nếu đã tính tay chắc chắn)


In [19]:
# 3) Ngẫu nhiên có seed, đúng ràng buộc: n in [1, 50], giá trị khác nhau trong [0, 1000], target in [0, 1000]
def random_bst_values(rng, n):
    """Chọn n giá trị khác nhau, chèn theo thứ tự ngẫu nhiên vào BST rồi xuất danh sách kiểu LeetCode."""
    vals = rng.sample(range(0, 1001), n)
    root = None
    for v in vals:
        root = insert_bst(root, v)
    return tree_to_list(root), sorted(vals)


rng = random.Random(776)
count = 0
for _ in range(3000):
    n = rng.randint(1, 50)
    values, srt = random_bst_values(rng, n)
    pick = rng.random()
    if pick < 0.4:
        target = rng.choice(srt)                      # bằng một giá trị có trong cây
    elif pick < 0.55:
        target = max(0, srt[0] - 1)                   # sát dưới giá trị nhỏ nhất
    elif pick < 0.7:
        target = min(1000, srt[-1])                   # bằng/lớn hơn giá trị lớn nhất
    else:
        target = rng.randint(0, 1000)
    for f, name in zip(IMPLS, NAMES):
        run_checked(f, values, target)
    count += 1
print(count, "cây ngẫu nhiên có seed: cả 5 cách qua bộ kiểm tra (BST, ngưỡng, đủ nút, quan hệ cha-con)")

# 4) Các cách tại chỗ (2, 3) cho CÙNG hình dạng; Cách 4 cũng vậy; Cách 1 có thể khác hình dạng khi có nút bị bỏ -> chỉ cần hợp lệ
same = 0
for _ in range(1000):
    n = rng.randint(1, 50)
    values, srt = random_bst_values(rng, n)
    target = rng.randint(0, 1000)
    r2 = run_checked(split_recursive, values, target)
    r3 = run_checked(split_iterative, values, target)
    r4 = run_checked(split_persistent, values, target)
    assert r2 == r3 == r4
    same += 1
print(same, "cây: đệ quy = lặp = persistent (cùng hình dạng)")

3000 cây ngẫu nhiên có seed: cả 5 cách qua bộ kiểm tra (BST, ngưỡng, đủ nút, quan hệ cha-con)


1000 cây: đệ quy = lặp = persistent (cùng hình dạng)


In [20]:
# 5) Cách 4 không phá cây gốc; Cách 2/3 phá (nên phải test trên bản sao); cây lệch sâu 50; đường cắt đi hết cây
base = build_tree(ex1)
split_persistent(base, 2)
assert tree_to_list(base) == ex1
print("persistent: cây gốc nguyên vẹn")

mutated = build_tree(ex1)
split_recursive(mutated, 2)
print("sau split tại chỗ, biến `mutated` chỉ còn:", tree_to_list(mutated))

chain = None
for v in range(1, 51):                      # cây lệch phải 1..50 (độ sâu 50)
    chain = insert_bst(chain, v)
for target in (0, 1, 25, 49, 50, 1000):
    for f, name in zip(IMPLS, NAMES):
        got = run_checked(f, tree_to_list(chain), target)
a, b = split_recursive(copy_tree(chain), 25)
assert inorder(a) == list(range(1, 26)) and inorder(b) == list(range(26, 51))
print("cây lệch 50 nút: tách đúng tại 25 ->", len(inorder(a)), "và", len(inorder(b)), "nút")

persistent: cây gốc nguyên vẹn
sau split tại chỗ, biến `mutated` chỉ còn: [4, 3, 6, None, None, 5, 7]
cây lệch 50 nút: tách đúng tại 25 -> 25 và 25 nút


## 13. Các lỗi người mới hay gặp

Trong mục này ta dùng chính `check_split` làm "giám khảo" để thấy lỗi bị bắt ở tính chất nào. Hàm `verdict` chạy một cách làm sai trên bản sao và in kết quả.

In [21]:
def verdict(fn, values, target):
    original = build_tree(values)
    try:
        result = fn(copy_tree(original), target)
        check_split(original, target, result)
        return "ĐÚNG (qua bộ kiểm tra)"
    except AssertionError as e:
        return f"SAI -> {e}"
    except Exception as e:                                    # lỗi chạy (vd: trả sai kiểu)
        return f"SAI (lỗi chạy: {type(e).__name__})"

### Lỗi 1: dùng `<` thay vì `<=` (nút bằng `target` rơi sai phần)

Đề: phần 1 gồm giá trị **nhỏ hơn hoặc bằng** `target`. Nếu viết `node.val < target` thì nút bằng `target` bị đẩy sang phần 2.

In [22]:
def wrong_strict(root, target):
    def dfs(node):
        if node is None:
            return [None, None]
        if node.val < target:                    # SAI: phải là <=
            l, r = dfs(node.right)
            node.right = l
            return [node, r]
        l, r = dfs(node.left)
        node.left = r
        return [l, node]

    return dfs(root)


print(verdict(wrong_strict, [1], 1), "<- ví dụ 2 của đề")
print(verdict(wrong_strict, [4, 2, 6, 1, 3, 5, 7], 2))
print(verdict(split_recursive, [1], 1))

SAI -> phần 2 có giá trị <= target <- ví dụ 2 của đề
SAI -> phần 2 có giá trị <= target
ĐÚNG (qua bộ kiểm tra)


**Vì sao sai:** với `[1]`, `target = 1`: `1 < 1` sai nên `1` sang phần 2, kết quả `[[], [1]]` thay vì `[[1], []]`. **Sửa:** `<=`.

### Lỗi 2: quên nối lại con trỏ (mất cây con đã cắt)

Cắt cây con phải bằng `l, r = dfs(node.right)` nhưng **không** gán `node.right = l` — `node.right` vẫn trỏ về cây cũ (còn lẫn nút lớn hơn target), hoặc ta lấy `r` mà bỏ rơi `l`.

In [23]:
def wrong_forgot_reattach(root, target):
    def dfs(node):
        if node is None:
            return [None, None]
        if node.val <= target:
            l, r = dfs(node.right)
            # SAI: thiếu  node.right = l   (node.right vẫn trỏ cây cũ chứa cả nút > target)
            return [node, r]
        l, r = dfs(node.left)
        # SAI: thiếu  node.left = r
        return [l, node]

    return dfs(root)


print(verdict(wrong_forgot_reattach, [4, 2, 6, 1, 3, 5, 7], 2))
print(verdict(wrong_forgot_reattach, [1], 1))

SAI -> phần 1 có giá trị > target
ĐÚNG (qua bộ kiểm tra)


**Vì sao sai:** sau khi tách, các nút của phần 2 vẫn còn "dính" trong phần 1 qua con trỏ cũ (và ngược lại) ⇒ trùng nút, sai ngưỡng. **Sửa:** luôn gán lại phía vừa cắt: `node.right = l` (trường hợp `≤`) hoặc `node.left = r` (trường hợp `>`).

### Lỗi 3: nối sai phía (mất cây con còn lại)

Trường hợp `node.val ≤ target`, ta cắt cây **phải** nhưng nối kết quả vào `node.left`: cây con trái gốc bị **ghi đè và mất**.

In [24]:
def wrong_side(root, target):
    def dfs(node):
        if node is None:
            return [None, None]
        if node.val <= target:
            l, r = dfs(node.right)
            node.left = l                         # SAI: phải là node.right = l
            node.right = None
            return [node, r]
        l, r = dfs(node.left)
        node.left = r
        return [l, node]

    return dfs(root)


print(verdict(wrong_side, [4, 2, 6, 1, 3, 5, 7], 3))
print(verdict(wrong_side, [4, 2, 6, 1, 3, 5, 7], 4))

SAI -> phần 1 không phải BST
SAI -> tập nút không khớp cây gốc (mất/thừa/trùng)


**Vì sao sai:** cây con trái của `node` (toàn giá trị `≤ target`, **phải giữ**) bị ghi đè ⇒ **mất nút** (hoặc phá BST). **Sửa:** cắt phía nào thì nối lại đúng phía đó.

### Lỗi 4: không cắt tiếp cây con còn lại ("chỉ xét gốc")

Xét `node.val ≤ target` rồi trả `[node, None]`: giả định cả cây gốc `node` thuộc phần 1. Nhưng cây con **phải** có thể chứa nút `> target`.

In [25]:
def wrong_root_only(root, target):
    if root is None:
        return [None, None]
    if root.val <= target:
        return [root, None]                        # SAI: cây con phải có thể còn nút > target
    return [None, root]                            # SAI: cây con trái có thể còn nút <= target


print(verdict(wrong_root_only, [4, 2, 6, 1, 3, 5, 7], 5))
print(verdict(wrong_root_only, [4, 2, 6, 1, 3, 5, 7], 2))
print(verdict(wrong_root_only, [1], 1))

SAI -> phần 1 có giá trị > target
SAI -> phần 2 có giá trị <= target
ĐÚNG (qua bộ kiểm tra)


**Vì sao sai:** chỉ so sánh gốc rồi gán cả cây cho một phần làm lẫn giá trị khác phe. **Sửa:** cắt đệ quy cây con phía còn có thể lẫn (mục 6).

### Lỗi 5: test bằng cây đã bị sửa (không sao chép trước)

Hàm tại chỗ **thay đổi** cây. Nếu test lấy "cây gốc" **sau khi** gọi hàm để so quan hệ cha–con, ta đang so kết quả với một cây **đã bị cắt**: bộ kiểm tra hoặc báo lỗi oan (như dưới đây) hoặc tệ hơn là bỏ sót lỗi thật. Hãy chứng kiến:

In [26]:
original = build_tree(ex1)
result = split_recursive(original, 2)               # SAI khi test: không sao chép trước
print("cây 'gốc' sau khi gọi:", tree_to_list(original), "<- đã không còn là cây gốc nữa")
try:
    check_split(original, 2, result)
    print("kiểm tra so với cây đã bị sửa: qua — nhưng đó là so với một cây đã bị cắt, nên không còn ý nghĩa")
except AssertionError as e:
    print("kiểm tra so với cây đã bị sửa:", e)

cây 'gốc' sau khi gọi: [4, 3, 6, None, None, 5, 7] <- đã không còn là cây gốc nữa
kiểm tra so với cây đã bị sửa: tập nút không khớp cây gốc (mất/thừa/trùng)


**Vì sao nguy hiểm:** cây "gốc" nay chỉ còn một nhánh của kết quả, nên nó không còn bằng chứng về cây ban đầu. **Sửa:** `original = build_tree(values)`; gọi hàm trên `copy_tree(original)` (như `run_checked` ở mục 12).

**Checklist chống lỗi:**

- [ ] Dùng `<=` cho phần 1.
- [ ] Cắt cây con **cùng phía** với phần chứa `node`... cụ thể: `≤` cắt **phải** nối `node.right = l`; `>` cắt **trái** nối `node.left = r`.
- [ ] Trả `[node, r]` (trường hợp `≤`) hoặc `[l, node]` (trường hợp `>`).
- [ ] Cơ sở `[None, None]`.
- [ ] Test bằng **tính chất** trên **bản sao** cây gốc.

## 14. Mẹo & nhận dạng pattern

- **"Tách/gộp BST theo một ngưỡng"** ⇒ **split/merge** kiểu treap: so `root.val` với ngưỡng rồi đệ quy **một phía**, nối lại con trỏ. Độ phức tạp `O(h)`.
- **Hàm trả về hai giá trị `[trái, phải]`** là mẫu gọn cho mọi bài "cắt cây thành hai phần".
- **Tính chất BST quyết định cả cây con:** so một nút là biết số phận của cả cây con phía "chắc chắn".
- **Giữ cấu trúc cũ tối đa ⇒ sửa tại chỗ, đi một đường.** Nếu phải giữ cây gốc ⇒ persistent (tạo nút mới trên đường cắt).
- **Test bài nhiều đáp án đúng:** viết hàm kiểm tra tính chất (BST, ngưỡng, đủ nút, quan hệ đề yêu cầu), không so cứng hình dạng; sao chép cây trước khi gọi hàm sửa tại chỗ.
- **Dummy node + đuôi** là mẹo biến đệ quy "nối kết quả" thành vòng lặp `O(1)` bộ nhớ.
- Bài họ hàng: 669 (Trim a BST), 450 (Delete Node in a BST), 701 (Insert into a BST), 98 (Validate BST), 1382 (Balance a BST), treap (split/merge).

## 15. Biến thể

1. **Tách ba phần** `< t`, `== t`, `> t`: `split(root, t - 1)` rồi tách phần sau bằng `split(·, t)` (với số nguyên).
2. **Gộp (merge)** hai BST mà mọi giá trị của cây trái nhỏ hơn mọi giá trị của cây phải: gắn cây phải vào con phải của nút lớn nhất cây trái (xem bài tập 5) — hoặc dùng treap để giữ cân bằng.
3. **Trim BST (bài 669):** giữ các nút trong `[low, high]` — hai lần cắt.
4. **Tách theo thứ hạng (rank):** `k` nút nhỏ nhất vào phần 1 (cần lưu kích thước cây con) — nền tảng của "implicit treap".
5. **Tách nhưng giữ cây gốc nguyên vẹn:** Cách 4 (persistent).

Minh hoạ biến thể 1 (tách ba phần) với bộ kiểm tra:

In [27]:
def split3(root, t):
    """Biến thể 1: [< t, == t, > t] cho giá trị nguyên."""
    small, rest = split_recursive(root, t - 1)       # small: <= t-1 ; rest: >= t
    equal, big = split_recursive(rest, t)            # equal: == t (<= t trong rest) ; big: > t
    return [small, equal, big]


orig = build_tree([4, 2, 6, 1, 3, 5, 7])
a, e, b = split3(copy_tree(orig), 4)
print(tree_to_list(a), tree_to_list(e), tree_to_list(b))
assert inorder(a) == [1, 2, 3] and inorder(e) == [4] and inorder(b) == [5, 6, 7]
a, e, b = split3(copy_tree(orig), 8)
assert inorder(a) == [1, 2, 3, 4, 5, 6, 7] and e is None and b is None
print("tách ba phần: OK")

[2, 1, 3] [4] [6, 5, 7]
tách ba phần: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng lại từ preorder | chia danh sách giá trị, chèn vào hai BST mới | `O(n · h)` | `O(n)` | oracle/hiểu bài; không phải cách tối ưu |
| 2. Đệ quy tại chỗ | cắt một phía, nối lại con trỏ, trả `[a, b]` | `O(h)` | `O(h)` | **nộp bài**, ngắn gọn |
| 3. Lặp, hai đuôi giả | đi một đường, nối vào đuôi phần 1 (`.right`) / phần 2 (`.left`) | `O(h)` | `O(1)` | tránh đệ quy, cây rất sâu |
| 4. Persistent | tạo nút mới chỉ trên đường cắt, chia sẻ cây con | `O(h)` | `O(h)` | cần giữ nguyên cây gốc / lịch sử |

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có cell TODO để bạn tự làm (cell **chưa** được gọi, chỉ gán `None` hoặc `def` rỗng). Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Với `root = [4,2,6,1,3,5,7]` và `target = 4`, hai cây trả về là gì? Ghi `bai1_du_doan = [danh_sach_phan_1, danh_sach_phan_2]` (dạng danh sách kiểu LeetCode, nhớ `None`).

In [28]:
# TODO Bài 1: bai1_du_doan = [[...], [...]]
bai1_du_doan = None

### Bài 2 (tính tay)

Cùng cây, `target = 5`. Viết hai danh sách kết quả vào `bai2 = [phan_1, phan_2]` (xem trace 8.2 nếu bí; hãy tự làm trước).

In [29]:
# TODO Bài 2: bai2 = [[...], [...]]
bai2 = None

### Bài 3 (điền TODO)

Hoàn thành `dfs` trong hàm `split_todo` (đã có khung).

In [30]:
def split_todo(root, target):
    def dfs(node):
        # TODO: node rỗng -> [None, None];
        # nếu node.val <= target: l, r = dfs(node.right); node.right = l; trả [node, r]
        # ngược lại: l, r = dfs(node.left); node.left = r; trả [l, node]
        pass

    return dfs(root)

### Bài 4 (tự viết lại)

Không nhìn mục 9, tự viết `split_bst(root, target)` (đệ quy, sửa tại chỗ).

In [31]:
# TODO Bài 4: tự viết lại
def split_bst(root, target):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `merge_bst(a, b)`: cho hai BST mà **mọi giá trị của `a` nhỏ hơn mọi giá trị của `b`**, trả về một BST chứa tất cả (gợi ý: gắn `b` vào con phải của nút lớn nhất của `a`). Rồi kiểm tra: tách một cây bằng `split_bst` rồi `merge_bst` hai phần lại phải cho BST có cùng dãy trung thứ tự với cây gốc.

In [32]:
# TODO Bài 5
def merge_bst(a, b):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `4 ≤ 4` ⇒ `4` ở phần 1 cùng cây con trái `2(1,3)`.
- Mức 2: cắt cây phải `6(5,7)` với `target = 4`: cả `6,5,7` đều `> 4`.
- Mức 3: `[[4, 2, None, 1, 3], [6, 5, 7]]` (phần 1: `4` với con trái `2`; phần 2: `6` với `5`, `7`).

**Bài 2**
- Mức 1: `4 ≤ 5` ⇒ giữ cây trái, cắt cây phải `6(5,7)`.
- Mức 2: `6 > 5` thuộc phần 2, cắt cây trái của `6` (nút `5 ≤ 5`).
- Mức 3: `5` nối làm con phải của `4`; `6` mất con trái ⇒ `[[4, 2, 5, 1, 3], [6, None, 7]]`.

**Bài 3**
- Mức 1: `if node is None: return [None, None]`.
- Mức 2: nhánh `<=`: `l, r = dfs(node.right)`.
- Mức 3: `node.right = l; return [node, r]`; nhánh còn lại đối xứng với `left`/`r`.

**Bài 4**
- Mức 1: hàm đệ quy trả hai giá trị.
- Mức 2: so `root.val` với `target`.
- Mức 3: giống mục 9.

**Bài 5**
- Mức 1: nếu `a` rỗng trả `b`; nếu `b` rỗng trả `a`.
- Mức 2: đi sang phải từ `a` đến nút không còn con phải.
- Mức 3: `cur.right = b`; trả `a`. Kiểm tra với `is_bst` và `inorder(kết quả) == inorder(a) + inorder(b)`.

## 19. Đáp án tham khảo

Chạy kiểm tra từng đáp án bằng `assert`.

In [33]:
# Bài 1
res = split_recursive(build_tree(ex1), 4)
dap_an_1 = [tree_to_list(r) for r in res]
print(dap_an_1)
assert dap_an_1 == [[4, 2, None, 1, 3], [6, 5, 7]]
assert run_checked(split_recursive, ex1, 4) == dap_an_1

# Bài 2
res = split_recursive(build_tree(ex1), 5)
bai2_dap_an = [tree_to_list(r) for r in res]
print(bai2_dap_an)
assert bai2_dap_an == [[4, 2, 5, 1, 3], [6, None, 7]]
assert run_checked(split_recursive, ex1, 5) == bai2_dap_an

# Bài 3
def split_todo(root, target):
    def dfs(node):
        if node is None:
            return [None, None]
        if node.val <= target:
            l, r = dfs(node.right)
            node.right = l
            return [node, r]
        l, r = dfs(node.left)
        node.left = r
        return [l, node]

    return dfs(root)


for values, target, _ in edge:
    run_checked(split_todo, values, target)
print("Bài 3: OK")

[[4, 2, None, 1, 3], [6, 5, 7]]
[[4, 2, 5, 1, 3], [6, None, 7]]
Bài 3: OK


In [34]:
# Bài 4
def split_bst(root, target):
    if root is None:
        return [None, None]
    if root.val <= target:
        l, r = split_bst(root.right, target)
        root.right = l
        return [root, r]
    l, r = split_bst(root.left, target)
    root.left = r
    return [l, root]


rng = random.Random(4)
for _ in range(1000):
    values, srt = random_bst_values(rng, rng.randint(1, 50))
    run_checked(split_bst, values, rng.randint(0, 1000))
print("Bài 4: OK")

# Bài 5
def merge_bst(a, b):
    if a is None:
        return b
    if b is None:
        return a
    cur = a
    while cur.right:
        cur = cur.right
    cur.right = b                         # b toàn giá trị lớn hơn mọi giá trị của a
    return a


rng = random.Random(5)
for _ in range(1000):
    values, srt = random_bst_values(rng, rng.randint(1, 50))
    target = rng.randint(0, 1000)
    root = build_tree(values)
    a, b = split_bst(copy_tree(root), target)
    merged = merge_bst(a, b)
    assert is_bst(merged) and inorder(merged) == inorder(root)
assert merge_bst(None, None) is None and inorder(merge_bst(build_tree([2, 1]), build_tree([5, None, 6]))) == [1, 2, 5, 6]
print("Bài 5: OK (tách rồi gộp giữ nguyên dãy trung thứ tự)")

Bài 4: OK


Bài 5: OK (tách rồi gộp giữ nguyên dãy trung thứ tự)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "tách BST theo ngưỡng" (hàm trả hai gốc, cắt một phía, nối lại):**

```python
class Solution:
    def splitBST(self, root: Optional[TreeNode], target: int) -> List[Optional[TreeNode]]:
        def dfs(root):
            if root is None:
                return [None, None]
            if root.val <= target:            # root thuộc phần 1, giữ nguyên cây con trái
                l, r = dfs(root.right)        # cắt cây con phải
                root.right = l                # nối phần <= target của cây phải
                return [root, r]
            else:                             # root thuộc phần 2, giữ nguyên cây con phải
                l, r = dfs(root.left)         # cắt cây con trái
                root.left = r                 # nối phần > target của cây trái
                return [l, root]

        return dfs(root)
```

**Checklist trước khi submit:**

- [ ] Phần 1 là `<= target` (không phải `<`)?
- [ ] `<=`: cắt cây **phải**, nối `root.right = l`, trả `[root, r]`; `>`: cắt cây **trái**, nối `root.left = r`, trả `[l, root]`?
- [ ] Cơ sở `[None, None]`, và có thể một trong hai phần rỗng?
- [ ] Không cố duyệt cả cây: chỉ đi **một đường** (`O(h)`)?
- [ ] Test bằng **hàm kiểm tra tính chất** (BST, ngưỡng, đủ nút, quan hệ cha–con) trên **bản sao** cây gốc?
- [ ] Đã đối chiếu lại đề gốc (bài Premium) nếu có tài khoản?